# Marketing table for customer analysis

- Add a RFM score to the table
- Create the final table
- Insert data into the final table

## Working query to aggregate the data

In [0]:
%skip
WITH base AS (
  SELECT
    c.customer_id,
    CONCAT(c.firstname, ' ', c.lastname) AS fullname,
    c.gender,
    c.birthdate,
    DATE_DIFF(year,c.birthdate, CURRENT_DATE()) AS age,
    c.country,
    MIN(s.order_date) AS first_order,
    MAX(s.order_date) AS last_order,
    COUNT(DISTINCT s.order_number) AS total_orders,
    SUM(s.sales) AS total_sales,
    SUM(s.quantity) AS total_quantity,
    ROUND(SUM(s.sales) / COUNT(DISTINCT s.order_number), 2) AS avg_order_value
  FROM data_lakehouse_project.gold.dim_customers AS c
  LEFT JOIN data_lakehouse_project.gold.fact_sales AS s
    ON c.customer_id = s.customer_id
  LEFT JOIN data_lakehouse_project.gold.dim_products AS p
    ON s.product_key = p.product_key
  GROUP BY ALL
),

top_category AS (
  SELECT customer_id, category, total_qty
  FROM (
    SELECT
      c.customer_id,
      p.category,
      SUM(s.quantity) AS total_qty,
      ROW_NUMBER() OVER(PARTITION BY c.customer_id ORDER BY SUM(s.quantity) DESC) AS rnk
    FROM data_lakehouse_project.gold.dim_customers AS c
    LEFT JOIN data_lakehouse_project.gold.fact_sales AS s 
      ON c.customer_id = s.customer_id
    LEFT JOIN data_lakehouse_project.gold.dim_products AS p 
      ON s.product_key = p.product_key
    GROUP BY ALL
  )
  WHERE rnk = 1
),

top_subcategory AS (
  SELECT customer_id, sub_category, total_qty
  FROM (
    SELECT
      c.customer_id,
      p.sub_category,
      SUM(s.quantity) AS total_qty,
      ROW_NUMBER() OVER(PARTITION BY c.customer_id ORDER BY SUM(s.quantity) DESC) AS rnk
    FROM data_lakehouse_project.gold.dim_customers AS c
    LEFT JOIN data_lakehouse_project.gold.fact_sales AS s 
      ON c.customer_id = s.customer_id
    LEFT JOIN data_lakehouse_project.gold.dim_products AS p 
      ON s.product_key = p.product_key
    GROUP BY ALL
  )
  WHERE rnk = 1
),

top_product AS (
  SELECT customer_id, product_name, total_qty
  FROM (
    SELECT
      c.customer_id,
      p.product_name,
      SUM(s.quantity) AS total_qty,
      ROW_NUMBER() OVER(PARTITION BY c.customer_id ORDER BY SUM(s.quantity) DESC) AS rnk
    FROM data_lakehouse_project.gold.dim_customers AS c
    LEFT JOIN data_lakehouse_project.gold.fact_sales AS s 
      ON c.customer_id = s.customer_id
    LEFT JOIN data_lakehouse_project.gold.dim_products AS p 
      ON s.product_key = p.product_key
    GROUP BY ALL
  )
  WHERE rnk = 1
),

rfm_scores AS (
  SELECT
    b.*,
    tc.category AS top_category,
    tsc.sub_category AS top_subcategory,
    tp.product_name AS top_product,
    DATEDIFF(b.last_order, MAX(b.last_order) OVER()) AS days_since_last_order,
    NTILE(5) OVER(ORDER BY b.last_order ASC) AS r_score,
    NTILE(5) OVER(ORDER BY b.total_orders ASC) AS f_score,
    NTILE(5) OVER(ORDER BY b.total_sales ASC) AS m_score,
    NTILE(5) OVER(ORDER BY b.avg_order_value ASC) AS v_score
  FROM base AS b
  LEFT JOIN top_category AS tc 
    ON b.customer_id = tc.customer_id
  LEFT JOIN top_subcategory AS tsc 
    ON b.customer_id = tsc.customer_id
  LEFT JOIN top_product AS tp 
    ON b.customer_id = tp.customer_id
)

SELECT  
  rfm.*,
  (r_score + f_score + m_score + v_score) / 4 AS rfmv_score,
  CASE
    WHEN rfmv_score >= 4.5 THEN 'VIP'
    WHEN rfmv_score >= 3.5 THEN 'Premium'
    WHEN rfmv_score >= 2.5 THEN 'Standard'
    WHEN rfmv_score >= 1.5 THEN 'At Risk'
    ELSE 'Dormant'
    END AS rfmv_segment
FROM rfm_scores AS rfm
ORDER BY rfmv_score 

## Create the table & Insert data

In [0]:
CREATE OR REPLACE TABLE data_lakehouse_project.platinium.customer_360 AS
WITH base AS (
  SELECT
    c.customer_id,
    CONCAT(c.firstname, ' ', c.lastname) AS fullname,
    c.gender,
    c.birthdate,
    DATE_DIFF(year,c.birthdate, CURRENT_DATE()) AS age,
    c.country,
    MIN(s.order_date) AS first_order,
    MAX(s.order_date) AS last_order,
    COUNT(DISTINCT s.order_number) AS total_orders,
    SUM(s.sales) AS total_sales,
    SUM(s.quantity) AS total_quantity,
    ROUND(SUM(s.sales) / COUNT(DISTINCT s.order_number), 2) AS avg_order_value
  FROM data_lakehouse_project.gold.dim_customers AS c
  LEFT JOIN data_lakehouse_project.gold.fact_sales AS s
    ON c.customer_id = s.customer_id
  LEFT JOIN data_lakehouse_project.gold.dim_products AS p
    ON s.product_key = p.product_key
  GROUP BY ALL
),

top_category AS (
  SELECT customer_id, category, total_qty
  FROM (
    SELECT
      c.customer_id,
      p.category,
      SUM(s.quantity) AS total_qty,
      ROW_NUMBER() OVER(PARTITION BY c.customer_id ORDER BY SUM(s.quantity) DESC) AS rnk
    FROM data_lakehouse_project.gold.dim_customers AS c
    LEFT JOIN data_lakehouse_project.gold.fact_sales AS s 
      ON c.customer_id = s.customer_id
    LEFT JOIN data_lakehouse_project.gold.dim_products AS p 
      ON s.product_key = p.product_key
    GROUP BY ALL
  )
  WHERE rnk = 1
),

top_subcategory AS (
  SELECT customer_id, sub_category, total_qty
  FROM (
    SELECT
      c.customer_id,
      p.sub_category,
      SUM(s.quantity) AS total_qty,
      ROW_NUMBER() OVER(PARTITION BY c.customer_id ORDER BY SUM(s.quantity) DESC) AS rnk
    FROM data_lakehouse_project.gold.dim_customers AS c
    LEFT JOIN data_lakehouse_project.gold.fact_sales AS s 
      ON c.customer_id = s.customer_id
    LEFT JOIN data_lakehouse_project.gold.dim_products AS p 
      ON s.product_key = p.product_key
    GROUP BY ALL
  )
  WHERE rnk = 1
),

top_product AS (
  SELECT customer_id, product_name, total_qty
  FROM (
    SELECT
      c.customer_id,
      p.product_name,
      SUM(s.quantity) AS total_qty,
      ROW_NUMBER() OVER(PARTITION BY c.customer_id ORDER BY SUM(s.quantity) DESC) AS rnk
    FROM data_lakehouse_project.gold.dim_customers AS c
    LEFT JOIN data_lakehouse_project.gold.fact_sales AS s 
      ON c.customer_id = s.customer_id
    LEFT JOIN data_lakehouse_project.gold.dim_products AS p 
      ON s.product_key = p.product_key
    GROUP BY ALL
  )
  WHERE rnk = 1
),

rfm_scores AS (
  SELECT
    b.*,
    tc.category AS top_category,
    tsc.sub_category AS top_subcategory,
    tp.product_name AS top_product,
    DATEDIFF(b.last_order, MAX(b.last_order) OVER()) AS days_since_last_order,
    NTILE(5) OVER(ORDER BY b.last_order ASC) AS r_score,
    NTILE(5) OVER(ORDER BY b.total_orders ASC) AS f_score,
    NTILE(5) OVER(ORDER BY b.total_sales ASC) AS m_score,
    NTILE(5) OVER(ORDER BY b.avg_order_value ASC) AS v_score
  FROM base AS b
  LEFT JOIN top_category AS tc 
    ON b.customer_id = tc.customer_id
  LEFT JOIN top_subcategory AS tsc 
    ON b.customer_id = tsc.customer_id
  LEFT JOIN top_product AS tp 
    ON b.customer_id = tp.customer_id
)

SELECT  
  rfm.*,
  (r_score + f_score + m_score + v_score) / 4 AS rfmv_score,
  CASE
    WHEN rfmv_score >= 4.5 THEN 'VIP'
    WHEN rfmv_score >= 3.5 THEN 'Premium'
    WHEN rfmv_score >= 2.5 THEN 'Standard'
    WHEN rfmv_score >= 1.5 THEN 'At Risk'
    ELSE 'Dormant'
    END AS rfmv_segment
FROM rfm_scores AS rfm
ORDER BY rfmv_score 

In [0]:
SELECT 
  *
FROM data_lakehouse_project.platinium.customer_360

